# SkyRL + Tinker: RL Training from Colab

Runs RL training using **Tinker** as the hosted GPU backend.  
Colab handles orchestration (CPU-only). Tinker handles distributed training.

**Resources:**
- SkyRL: https://github.com/NovaSky-AI/SkyRL
- Tinker docs: https://tinker-docs.thinkingmachines.ai
- Tinker console (get API key): https://console.thinkingmachines.ai
- Harbor Framework: https://harborframework.com/docs/training-workflows/rl

## 1. Install dependencies

In [ ]:
# Core Tinker SDK
!pip install tinker tinker_cookbook

# SkyRL (CPU-only install — Colab only runs the orchestration loop)
!git clone https://github.com/NovaSky-AI/SkyRL.git
%cd SkyRL

# Install skyrl-train with Tinker + FSDP extras (CPU wheels only for Colab)
!pip install -e skyrl-train[tinker] --extra-index-url https://download.pytorch.org/whl/cpu
!pip install -e skyrl-gym

## 2. Set Tinker API key

Get your API key from https://console.thinkingmachines.ai  
Use Colab secrets (recommended) or paste directly.

In [ ]:
import os

# Option A: Colab Secrets (recommended — add TINKER_API_KEY in the key icon on the left)
try:
    from google.colab import userdata
    os.environ["TINKER_API_KEY"] = userdata.get("TINKER_API_KEY")
    print("Loaded API key from Colab Secrets")
except Exception:
    # Option B: Paste directly (not recommended for shared notebooks)
    os.environ["TINKER_API_KEY"] = "your_api_key_here"  # <-- replace
    print("Using hardcoded API key")

## 3. Connect to Tinker

In [ ]:
import tinker

# base_url=None uses Tinker's cloud service
service_client = tinker.ServiceClient(base_url=None)

# Verify connection
print("Connected to Tinker:", service_client)

## 4. Configure the model and LoRA adapter

In [ ]:
MODEL_NAME = "Qwen/Qwen3-8B"  # or Qwen/Qwen3-4B-Instruct-2507 for faster iteration
LORA_RANK = 32

training_client = service_client.create_lora_training_client(
    base_model=MODEL_NAME,
    rank=LORA_RANK,
)
print(f"Training client ready: model={MODEL_NAME}, lora_rank={LORA_RANK}")

## 5a. Option A — SkyRL Tinker API server (self-hosted on Tinker GPUs)

SkyRL's `skyrl-tx` package launches a Tinker-compatible API server.  
This is the path if you want to use SkyRL's distributed PPO/GRPO trainer.

In [ ]:
# Launch the SkyRL Tinker API server
# This runs on Tinker's GPU infra — Colab just triggers it
import subprocess, threading

def launch_skyrl_tinker():
    subprocess.run(
        ["python", "-m", "skyrl.tinker.api"],
        cwd="/content/SkyRL/skyrl-train",
        env=os.environ,
    )

# Start in background thread
t = threading.Thread(target=launch_skyrl_tinker, daemon=True)
t.start()
print("SkyRL Tinker API server starting...")

## 5b. Option B — rLLM Tinker backend (simpler, recommended for starters)

Use rLLM's `AgentTrainer` with `backend="tinker"` — no local GPU needed.

In [ ]:
!pip install rllm[tinker] --torch-backend=cpu

In [ ]:
# Example: GRPO training on a math reasoning task
from rllm.trainer import AgentTrainer
from rllm.data.dataset import DatasetRegistry
from datasets import load_dataset

# Load a small dataset (GSM8K for quick test)
raw = load_dataset("openai/gsm8k", "main", split="train[:256]")

def gsm8k_reward(response, ground_truth):
    """Simple exact-match reward."""
    answer = ground_truth.split("#### ")[-1].strip()
    return 1.0 if answer in response else 0.0

config = {
    "tinker_base_url": None,  # None = Tinker cloud
    "model": {
        "name": MODEL_NAME,
        "lora_rank": LORA_RANK,
        "train_attn": True,
        "train_mlp": True,
    },
    "training": {
        "group_size": 8,
        "learning_rate": 2e-5,
        "max_length": 4096,
    },
    "sampling": {"temperature": 0.7, "top_p": 0.95},
    "algorithm": {
        "adv_estimator": "grpo",
        "norm_adv_by_std_in_grpo": True,
        "grouping_level": "trajectory",
    },
    "data": {
        "train_batch_size": 32,
        "max_prompt_length": 512,
        "max_response_length": 1024,
    },
    "trainer": {
        "total_epochs": 5,
        "logger": ["console"],
        "project_name": "skyrl-tinker-colab",
        "experiment_name": "gsm8k-grpo",
        "test_freq": 2,
        "save_freq": 5,
        "default_local_dir": "/content/checkpoints",
    },
}

print("Config ready. Run trainer.train() in the next cell.")

## 6. Run RL training

In [ ]:
# NOTE: This sends training jobs to Tinker's GPU cloud.
# Monitor progress at https://console.thinkingmachines.ai

trainer = AgentTrainer(
    config=config,
    train_dataset=raw,
    reward_fn=gsm8k_reward,
    backend="tinker",
)

trainer.train()

## 7. Harbor tasks (advanced)

Run RL on Harbor's containerized agentic tasks via SkyRL + Tinker.  
See: https://harborframework.com/docs/training-workflows/rl

In [ ]:
!pip install harbor-sdk  # Harbor Python SDK

In [ ]:
from harbor.job import Job
from harbor.models.job.config import JobConfig, OrchestratorConfig
from harbor.models.trial.config import EnvironmentConfig, TaskConfig
from harbor.models.environment_type import EnvironmentType
from pathlib import Path

# Define Harbor tasks to train on
task_configs = [
    TaskConfig(path="cancel-async-tasks"),
    TaskConfig(path="fix-dockerfile-syntax"),
]

# Rollout interface wired to Tinker model
class TinkerHarborRollout:
    def __init__(self, tinker_base_url, model_name):
        self.base_url = tinker_base_url
        self.model_name = model_name

    async def run(self, task_ids):
        job = Job(
            config=JobConfig(
                jobs_dir=Path("jobs"),
                environment=EnvironmentConfig(type=EnvironmentType.LOCAL),
                orchestrator=OrchestratorConfig(n_concurrent_trials=8),
                tasks=task_configs,
            )
        )
        result = await job.run()
        rollouts = []
        for trial in result.trial_results:
            reward = (
                trial.verifier_result.rewards.get("reward", 0)
                if trial.verifier_result else 0
            )
            token_ids = trial.agent_result.metadata["token_ids"]
            mask_ids = trial.agent_result.metadata["mask_ids"]
            rollouts.append({"reward": reward, "token_ids": token_ids, "mask_ids": mask_ids})
        return rollouts

rollout = TinkerHarborRollout(
    tinker_base_url=None,  # None = Tinker cloud
    model_name=MODEL_NAME,
)
print("Harbor rollout interface ready")

## 8. Download trained weights

In [ ]:
# List saved checkpoints
!tinker model list

# Download a checkpoint (replace <model-id> with actual ID)
# !tinker model download <model-id> --output /content/model_weights

## Useful Tinker CLI commands

```bash
tinker model list                        # list checkpoints
tinker model download <id>               # download weights
tinker job list                          # list training jobs
tinker job logs <id>                     # stream logs
```

Monitor training: https://console.thinkingmachines.ai